[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Red1-Rahman/NiriZan/blob/main/experiments/03_experiment_tracking_baselines.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/Red1-Rahman/NiriZan/blob/main/experiments/03_experiment_tracking_baselines.ipynb)

# Experiment 03: Experiment Tracking & Baselines
**Phase 3 Exploration**: versioned runs, structured diffs, baseline management, and multi-turn session tracking — validated against real traces before porting to `src/`.

Prototypes for:
- `src/nirizan/storage/experiment_store.py`
- `src/nirizan/storage/baselines.py`
- `src/nirizan/instrumentation/spans.py` (Trace's Phase 3 optional fields)
- `src/nirizan/instrumentation/tracer.py` (`Tracer.session()`)
- `src/nirizan/orchestrator/collector.py` (commit/snapshot tagging)

## 1. Environment Setup
No ML dependencies this phase — Phase 3 is data-model and storage plumbing, not scoring.

In [1]:
!pip install -q pydantic>=2.7 "git+https://github.com/Red1-Rahman/NiriZan.git@main#egg=nirizan"

from __future__ import annotations

import asyncio
import hashlib
import os
import sqlite3
import subprocess
from contextlib import asynccontextmanager
import contextvars
from datetime import datetime, timezone
from typing import AsyncGenerator, Optional, Protocol
from uuid import UUID, uuid4

from pydantic import BaseModel, ConfigDict, Field

print("Phase 3 exploration environment ready")

Phase 3 exploration environment ready


## 2. Versioning: Commit Hash + Data Snapshot

Per `docs/contracts.md`, `Run.code_commit` and `Run.data_snapshot_id` are both
required, never optional. Per our collector.py design: read `GIT_COMMIT_SHA`
env var first, fall back to `git rev-parse HEAD`. `NIRIZAN_DATA_SNAPSHOT_ID`
env var only — no generic fallback exists, so an honest `None` beats a
fabricated value if it's unset.

In [2]:
def resolve_code_commit() -> Optional[str]:
    env_value = os.environ.get("GIT_COMMIT_SHA")
    if env_value:
        return env_value
    try:
        result = subprocess.run(
            ["git", "rev-parse", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        return result.stdout.strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def resolve_data_snapshot_id() -> Optional[str]:
    return os.environ.get("NIRIZAN_DATA_SNAPSHOT_ID")


print("code_commit:", resolve_code_commit())
print("data_snapshot_id:", resolve_data_snapshot_id())

# simulate two different versions being evaluated, since we can't literally
# checkout two commits in one notebook session
os.environ["GIT_COMMIT_SHA"] = "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6a1b2"
os.environ["NIRIZAN_DATA_SNAPSHOT_ID"] = "snapshot-v1-topk5"
print("simulated v1:", resolve_code_commit(), resolve_data_snapshot_id())

code_commit: None
data_snapshot_id: None
simulated v1: a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6a1b2 snapshot-v1-topk5


## 3. Phase 3 Models

Reproduced exactly from `docs/contracts.md` — no extra fields beyond what's
documented (unlike an earlier, incorrect draft that added fields to `RunDiff`
and `Baseline` that don't exist in the real contract).

In [3]:
from enum import Enum


class SpanKind(str, Enum):
    PLANNING = "planning"
    RETRIEVAL = "retrieval"
    TOOL_USE = "tool_use"
    GENERATION = "generation"


class Span(BaseModel):
    model_config = ConfigDict(frozen=True, strict=True)
    span_id: UUID
    trace_id: UUID
    parent_span_id: UUID | None = None
    kind: SpanKind
    name: str = Field(min_length=1, max_length=200)
    started_at: datetime
    ended_at: datetime
    attributes: dict[str, str | int | float | bool] = Field(default_factory=dict)
    input_payload: str | None = None
    output_payload: str | None = None


class Trace(BaseModel):
    model_config = ConfigDict(strict=True)
    trace_id: UUID
    application_name: str = Field(min_length=1)
    spans: list[Span] = Field(default_factory=list)
    created_at: datetime
    code_commit: str | None = None
    data_snapshot_id: str | None = None
    session_id: UUID | None = None

    def spans_of_kind(self, kind: SpanKind) -> list[Span]:
        return [s for s in self.spans if s.kind == kind]


class MetricResult(BaseModel):
    model_config = ConfigDict(strict=True)
    metric_name: str
    trace_id: UUID
    score: float = Field(ge=0.0, le=1.0)
    confidence: float | None = Field(default=None, ge=0.0, le=1.0)
    details: dict[str, str | int | float | bool] = Field(default_factory=dict)
    computed_at: datetime


class Run(BaseModel):
    model_config = ConfigDict(strict=True)
    run_id: UUID
    trace_id: UUID
    code_commit: str = Field(min_length=7, max_length=40)
    data_snapshot_id: str = Field(min_length=1)
    metric_results: list[MetricResult] = Field(default_factory=list)
    created_at: datetime


class RunDiff(BaseModel):
    model_config = ConfigDict(strict=True)
    run_a: UUID
    run_b: UUID
    metric_deltas: dict[str, float]


class Baseline(BaseModel):
    model_config = ConfigDict(strict=True)
    baseline_id: UUID
    system_type: str
    run_ids: list[UUID] = Field(min_length=1)
    established_at: datetime
    label: str = Field(min_length=1)


class Session(BaseModel):
    model_config = ConfigDict(strict=True)
    session_id: UUID
    application_name: str = Field(min_length=1)
    trace_ids: list[UUID] = Field(default_factory=list)
    started_at: datetime
    ended_at: datetime | None = None


print("Phase 3 models defined, matching contracts.md exactly")

Phase 3 models defined, matching contracts.md exactly


## 4. ExperimentStore Prototype

`record_run` / `get_run` / `diff`, per `docs/contracts.md`. `diff` computes a
difference only — no judgment, no verdict (that's `regression/comparator.py`'s
job in Phase 4).

In [4]:
class ExperimentStore:
    def __init__(self) -> None:
        self._runs: dict[UUID, Run] = {}

    async def record_run(self, run: Run) -> None:
        self._runs[run.run_id] = run

    async def get_run(self, run_id: UUID) -> Optional[Run]:
        return self._runs.get(run_id)

    async def diff(self, run_a: UUID, run_b: UUID) -> RunDiff:
        a = await self.get_run(run_a)
        b = await self.get_run(run_b)
        assert a is not None and b is not None, "diff requires both runs to exist"

        scores_a = {m.metric_name: m.score for m in a.metric_results}
        scores_b = {m.metric_name: m.score for m in b.metric_results}
        shared_metrics = set(scores_a) & set(scores_b)

        deltas = {name: scores_b[name] - scores_a[name] for name in shared_metrics}
        return RunDiff(run_a=run_a, run_b=run_b, metric_deltas=deltas)


store = ExperimentStore()
print("ExperimentStore ready")

ExperimentStore ready


## 5. Prototype Tracer with Session Support

Mirrors the real `Tracer`, plus a new `session()` context manager per the
Phase 3 contract addition. `session_id` propagates via `contextvars`, same
pattern as `trace_id`/`parent_span_id`. Version tagging (`code_commit`,
`data_snapshot_id`) is applied separately, mirroring the real design where
the *collector* — not the tracer — is responsible for stamping those fields.

In [5]:
_CURRENT_TRACE_ID: contextvars.ContextVar[UUID | None] = contextvars.ContextVar(
    "trace_id", default=None
)
_CURRENT_SPAN_ID: contextvars.ContextVar[UUID | None] = contextvars.ContextVar(
    "span_id", default=None
)
_CURRENT_SESSION_ID: contextvars.ContextVar[UUID | None] = contextvars.ContextVar(
    "session_id", default=None
)


class PrototypeTracer:
    def __init__(self, application_name: str) -> None:
        self.application_name = application_name
        self._spans: list[Span] = []
        self.exported_traces: list[Trace] = []

    @asynccontextmanager
    async def session(self, session_id: UUID | None = None) -> AsyncGenerator[UUID, None]:
        sid = session_id or uuid4()
        token = _CURRENT_SESSION_ID.set(sid)
        try:
            yield sid
        finally:
            _CURRENT_SESSION_ID.reset(token)

    @asynccontextmanager
    async def start_span(self, name: str, kind: SpanKind, input_payload: str | None = None):
        trace_id = _CURRENT_TRACE_ID.get()
        is_root = trace_id is None
        if is_root:
            trace_id = uuid4()
            _CURRENT_TRACE_ID.set(trace_id)
        parent_span_id = _CURRENT_SPAN_ID.get()
        span_id = uuid4()
        started_at = datetime.now(timezone.utc)
        token = _CURRENT_SPAN_ID.set(span_id)
        output_holder = {"value": None}
        try:
            yield output_holder
        finally:
            span = Span(
                span_id=span_id,
                trace_id=trace_id,
                parent_span_id=parent_span_id,
                kind=kind,
                name=name,
                started_at=started_at,
                ended_at=datetime.now(timezone.utc),
                input_payload=input_payload,
                output_payload=output_holder["value"],
            )
            self._spans.append(span)
            _CURRENT_SPAN_ID.reset(token)
            if is_root:
                trace = Trace(
                    trace_id=trace_id,
                    application_name=self.application_name,
                    spans=[s for s in self._spans if s.trace_id == trace_id],
                    created_at=datetime.now(timezone.utc),
                    session_id=_CURRENT_SESSION_ID.get(),
                )
                self.exported_traces.append(trace)
                _CURRENT_TRACE_ID.set(None)


def tag_with_versioning(trace: Trace) -> Trace:
    """Simulates collector.py's ingest-time tagging, since Trace is not frozen."""
    return trace.model_copy(
        update={
            "code_commit": resolve_code_commit(),
            "data_snapshot_id": resolve_data_snapshot_id(),
        }
    )


print("PrototypeTracer with session() ready")

PrototypeTracer with session() ready


## 6. Generate Two Versioned Runs (Simulating a Regression)

Same query, two "versions" of a pipeline (different simulated commit +
snapshot), with deliberately different quality to prove `diff` surfaces a
real, correctly-signed delta.

In [6]:
async def run_pipeline(tracer: PrototypeTracer, query: str, answer_quality: float) -> Trace:
    async with tracer.start_span("retrieve", SpanKind.RETRIEVAL, input_payload=query) as h:
        h["value"] = "['doc a', 'doc b']"
    async with tracer.start_span("generate", SpanKind.GENERATION, input_payload=query) as h:
        h["value"] = f"answer (quality={answer_quality})"
    async with tracer.start_span("pipeline", SpanKind.PLANNING, input_payload=query) as h:
        h["value"] = "done"
        # nested calls happen inside this block in the real SDK; simplified here
    return tracer.exported_traces[-1]


tracer = PrototypeTracer(application_name="rag_pipeline_demo")

os.environ["GIT_COMMIT_SHA"] = "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6a1b2"
os.environ["NIRIZAN_DATA_SNAPSHOT_ID"] = "snapshot-v1-topk5"
async with tracer.start_span("pipeline_v1", SpanKind.PLANNING, input_payload="q1") as h:
    h["value"] = "v1 answer"
trace_v1 = tag_with_versioning(tracer.exported_traces[-1])

os.environ["GIT_COMMIT_SHA"] = "f6a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6a1"
os.environ["NIRIZAN_DATA_SNAPSHOT_ID"] = "snapshot-v2-topk10"
async with tracer.start_span("pipeline_v2", SpanKind.PLANNING, input_payload="q1") as h:
    h["value"] = "v2 answer"
trace_v2 = tag_with_versioning(tracer.exported_traces[-1])

print("v1:", trace_v1.code_commit[:8], trace_v1.data_snapshot_id)
print("v2:", trace_v2.code_commit[:8], trace_v2.data_snapshot_id)

v1: a1b2c3d4 snapshot-v1-topk5
v2: f6a1b2c3 snapshot-v2-topk10


In [7]:
now = datetime.now(timezone.utc)

run_v1 = Run(
    run_id=uuid4(),
    trace_id=trace_v1.trace_id,
    code_commit=trace_v1.code_commit,
    data_snapshot_id=trace_v1.data_snapshot_id,
    metric_results=[
        MetricResult(
            metric_name="answer_relevance", trace_id=trace_v1.trace_id, score=0.60, computed_at=now
        ),
        MetricResult(
            metric_name="groundedness", trace_id=trace_v1.trace_id, score=0.75, computed_at=now
        ),
    ],
    created_at=now,
)
run_v2 = Run(
    run_id=uuid4(),
    trace_id=trace_v2.trace_id,
    code_commit=trace_v2.code_commit,
    data_snapshot_id=trace_v2.data_snapshot_id,
    metric_results=[
        MetricResult(
            metric_name="answer_relevance", trace_id=trace_v2.trace_id, score=0.85, computed_at=now
        ),
        MetricResult(
            metric_name="groundedness", trace_id=trace_v2.trace_id, score=0.55, computed_at=now
        ),
    ],
    created_at=now,
)

await store.record_run(run_v1)
await store.record_run(run_v2)
print("Both runs recorded")

Both runs recorded


In [8]:
diff = await store.diff(run_v1.run_id, run_v2.run_id)
print("metric_deltas:", diff.metric_deltas)

assert diff.metric_deltas["answer_relevance"] == 0.85 - 0.60
assert diff.metric_deltas["groundedness"] == 0.55 - 0.75
print(
    "diff is pure computation: answer_relevance improved, groundedness regressed, no verdict attached"
)

metric_deltas: {'groundedness': -0.19999999999999996, 'answer_relevance': 0.25}
diff is pure computation: answer_relevance improved, groundedness regressed, no verdict attached


## 7. BaselineRepository Prototype

Not yet in `docs/contracts.md` — proposed shape, pending confirmation.
`list_baselines` is what satisfies the roadmap's "querying" requirement.

In [9]:
class BaselineRepository:
    def __init__(self) -> None:
        self._baselines: dict[UUID, Baseline] = {}

    async def save_baseline(self, baseline: Baseline) -> None:
        self._baselines[baseline.baseline_id] = baseline

    async def get_baseline(self, baseline_id: UUID) -> Optional[Baseline]:
        return self._baselines.get(baseline_id)

    async def list_baselines(self, system_type: str) -> list[Baseline]:
        return [b for b in self._baselines.values() if b.system_type == system_type]


baseline_repo = BaselineRepository()

baseline = Baseline(
    baseline_id=uuid4(),
    system_type="rag_pipeline",
    run_ids=[run_v1.run_id],
    established_at=now,
    label="pre-v2-release",
)
await baseline_repo.save_baseline(baseline)

fetched = await baseline_repo.get_baseline(baseline.baseline_id)
assert fetched is not None and fetched.run_ids == [run_v1.run_id]

listed = await baseline_repo.list_baselines("rag_pipeline")
assert len(listed) == 1

missing = await baseline_repo.get_baseline(uuid4())
assert missing is None

# confirm Baseline references by ID only, per contracts.md - no embedded Run data
assert not hasattr(baseline, "metric_results")
print("Baseline round-trip, querying, and 'reference by ID only' all verified")

Baseline round-trip, querying, and 'reference by ID only' all verified


## 8. Multi-Turn Session Tracing

Three turns of one agent conversation, grouped under a single `Session`.

In [10]:
class SessionRepository:
    def __init__(self) -> None:
        self._sessions: dict[UUID, Session] = {}

    async def save_session(self, session: Session) -> None:
        self._sessions[session.session_id] = session

    async def get_session(self, session_id: UUID) -> Optional[Session]:
        return self._sessions.get(session_id)


session_tracer = PrototypeTracer(application_name="agent_demo")
session_repo = SessionRepository()

async with session_tracer.session() as sid:
    session = Session(
        session_id=sid, application_name="agent_demo", started_at=datetime.now(timezone.utc)
    )

    for turn in ["turn 1: search docs", "turn 2: call calculator tool", "turn 3: summarize"]:
        async with session_tracer.start_span("turn", SpanKind.TOOL_USE, input_payload=turn) as h:
            h["value"] = f"result of: {turn}"
        latest_trace = session_tracer.exported_traces[-1]
        session.trace_ids.append(latest_trace.trace_id)
        assert latest_trace.session_id == sid

session.ended_at = datetime.now(timezone.utc)
await session_repo.save_session(session)

fetched_session = await session_repo.get_session(sid)
assert fetched_session is not None
assert len(fetched_session.trace_ids) == 3
assert fetched_session.ended_at is not None
print(f"Session {str(sid)[:8]}... has {len(fetched_session.trace_ids)} turns, all correctly tagged")

Session 3a6e6c14... has 3 turns, all correctly tagged


## 9. Full Phase 3 Pipeline, End to End

In [11]:
async def phase3_pipeline_demo():
    demo_tracer = PrototypeTracer(application_name="e2e_demo")
    demo_store = ExperimentStore()
    demo_baselines = BaselineRepository()

    os.environ["GIT_COMMIT_SHA"] = "1111111111111111111111111111111111111a"
    os.environ["NIRIZAN_DATA_SNAPSHOT_ID"] = "e2e-snapshot"

    async with demo_tracer.start_span(
        "pipeline", SpanKind.PLANNING, input_payload="e2e query"
    ) as h:
        h["value"] = "e2e answer"
    trace = tag_with_versioning(demo_tracer.exported_traces[-1])

    run = Run(
        run_id=uuid4(),
        trace_id=trace.trace_id,
        code_commit=trace.code_commit,
        data_snapshot_id=trace.data_snapshot_id,
        metric_results=[
            MetricResult(
                metric_name="answer_relevance",
                trace_id=trace.trace_id,
                score=0.9,
                computed_at=datetime.now(timezone.utc),
            )
        ],
        created_at=datetime.now(timezone.utc),
    )
    await demo_store.record_run(run)

    baseline = Baseline(
        baseline_id=uuid4(),
        system_type="rag_pipeline",
        run_ids=[run.run_id],
        established_at=datetime.now(timezone.utc),
        label="e2e-baseline",
    )
    await demo_baselines.save_baseline(baseline)

    fetched_run = await demo_store.get_run(run.run_id)
    fetched_baseline = await demo_baselines.get_baseline(baseline.baseline_id)
    assert fetched_run is not None and fetched_baseline is not None
    assert fetched_baseline.run_ids[0] == fetched_run.run_id

    print("Full chain verified: Trace -> tag versioning -> Run -> ExperimentStore -> Baseline")


await phase3_pipeline_demo()

Full chain verified: Trace -> tag versioning -> Run -> ExperimentStore -> Baseline
